# Audio Coding seminar WS 2026/27 – starter kit demo

Run this notebook once (in Colab: *Runtime → Run all*). It clones the course coder (Python-Audio-Coder, at the version the kit was tested with) and measures a few rate–quality points on the shared test set, the way every group reports them.

**Before you start:** nothing to do if you opened this notebook from the kit's GitHub page (*Open in Colab*): the first cell downloads the kit. If you use `starter_kit.zip` from Moodle instead, upload it to the Files panel on the left first (or set `KIT_ZIP` below to its path).

**Coming back in a later Colab session?** Run the first cell, then the last cell (your group's GitLab repository): it clones your repository and you continue from there.

In [ ]:
KIT_ZIP = 'starter_kit.zip'   # starter kit zip from Moodle, if you uploaded one
KIT_REPO = 'https://github.com/TUIlmenauAMS/ac-ws2627-starter-kit'   # used when there is no zip
import os, subprocess, sys
import zipfile
if not os.path.isdir('starter_kit'):
    if os.path.isfile(KIT_ZIP):
        if not zipfile.is_zipfile(KIT_ZIP):
            raise SystemExit('starter_kit.zip is incomplete: wait until the upload has finished '
                             '(progress circle at the bottom of the Files panel), then run this cell again.')
        zipfile.ZipFile(KIT_ZIP).extractall('.')
    else:
        subprocess.run(['git', 'clone', '-q', '--depth', '1', KIT_REPO, 'starter_kit'], check=True)
        cloned_kit = True
sys.path.insert(0, os.path.abspath('starter_kit'))
import setup_topic
if globals().get('cloned_kit'):
    # plain files: the kit goes into your group's GitLab repository, not as a nested repository
    setup_topic.remove_tree(os.path.join('starter_kit', '.git'))
# the course coder at the commit the kit was tested with (it includes the fixes, see starter_kit/README.md)
print(setup_topic.clone('Python-Audio-Coder')[1])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dahuffman'], check=True)
os.environ['AC_CODER_DIR'] = os.path.abspath('Python-Audio-Coder')
import ac_kit as k
print('ready, kit version', k.KIT_VERSION)

## The shared test set
The test set is cut from the **EBU SQAM** recordings (the standard material for testing audio coders). They may be used for non-commercial research and teaching but not redistributed, so the next cell downloads them from the EBU (175.5 MB, under a minute) and cuts the same 17 clips for everyone, checked bit for bit.

By running it you accept the EBU condition: *non-commercial R&D use only*. Do not put the clips in your GitLab repository; your README points to this download step instead.

If the EBU download fails, the cell falls back to the kit's own clips (`librosa-v1`) and says so in a warning. Results on the two test sets are not comparable, so the id is stored in every results row: report it in your paper.

Already have the SQAM zip (e.g. on Google Drive)? Set `SQAM_ZIP` to its path to skip the download.

In [ ]:
SQAM_ZIP = None   # or a path such as '/content/drive/MyDrive/TECH3253_SQAM_FLAC.zip'
TESTSET = k.prepare_test_set(sqam_zip=SQAM_ZIP)
TESTSET

In [ ]:
import pandas as pd
pd.DataFrame(k.load_manifest())[['file','class','split','channels','duration_s','attribution']]

## A rate–quality sweep
`quality_sweep` encodes and decodes every clip at every quality value, measures the bitrate from the **file size**, removes the coder delay (1024 samples) and computes SNR and the psychoacoustic loss. Results go to a CSV file: keep it next to your figures.

In [ ]:
clips = k.load_manifest(split='tuning', classes=['speech', 'polyphonic'])
rows = k.quality_sweep(clips, qualities=(50, 100, 200, 400), out_csv='demo_results.csv', method='baseline (sine window)', append=False)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
k.plot_rate_quality('demo_results.csv', 'snr_db', ax=ax[0])
k.plot_rate_quality('demo_results.csv', 'percloss', ax=ax[1])
plt.tight_layout(); plt.savefig('demo_rate_quality.png', dpi=150)

## Check your setup
Your numbers must match the reference for your test set (`starter_kit/reference_results_<test set>.csv`, same clips, same quality values). If they don't, your coder or measurement differs from everyone else's: fix that before you start.

In [ ]:
ref = pd.read_csv('starter_kit/reference_results_%s.csv' % TESTSET)
mine = pd.read_csv('demo_results.csv')
m = mine.merge(ref, on=['clip', 'quality'], suffixes=('', '_ref'))
assert len(m) == len(mine), 'some of your clips are not in the reference'
m['bitrate_diff'] = (m.bits_per_sample - m.bits_per_sample_ref).abs()
m['snr_diff'] = (m.snr_db - m.snr_db_ref).abs()
print(m[['clip', 'quality', 'bits_per_sample', 'bits_per_sample_ref', 'snr_db', 'snr_db_ref']])
assert (m.bitrate_diff < 1e-3).all() and (m.snr_diff < 0.01).all(), 'your setup differs from the reference'
print('setup matches the reference for test set', TESTSET)

## Using the functions in your own code
```python
fs, x = k.read_wav('starter_kit/testset/speech_2.wav')
out = k.run_coder('starter_kit/testset/speech_2.wav', quality=100)        # or coder_dir='my_modified_coder'
fs, y = k.read_wav(out['decoded'])
k.bits_per_sample(out['compressed'], k.num_samples(x))                    # bitrate from the file size
k.evaluate(x, y, fs, delay=1024)                                          # {'snr_db', 'percloss', 'compared'}
k.lossless_check(x, y_lossless, delay=0, trim_end=10)                     # lossless topics
```
See `starter_kit/README.md` for details and topic-specific notes.

## Your topic's baseline (for your proposal)
Set your topic number and variant, then run the cell. It first sets up the other course code your topic needs (`starter_kit/setup_topic.py`), then runs the unmodified coder on the tuning clips your topic needs (lossless topics: the lossless coders; Topic 15: Opus on LibriSpeech, which needs `pip install pesq pystoi`) and writes `results/baseline.csv`. The last cell puts it into your GitLab repository. `starter_kit/START_HERE.md` says what else your topic needs.

In [ ]:
TOPIC = 1        # your topic number, 1-15
VARIANT = 'a'    # your variant, 'a' or 'b'
print(subprocess.run([sys.executable, 'starter_kit/setup_topic.py', '--topic', str(TOPIC)],
                     capture_output=True, text=True, check=True).stdout)
r = subprocess.run([sys.executable, 'starter_kit/baseline.py', '--topic', str(TOPIC), '--variant', VARIANT,
                    '--out', 'results/baseline.csv'], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError('baseline.py failed:\n' + r.stderr[-3000:])

## Your group's GitLab repository
**First time** (one member, after the baseline cell): create your group's repository first (Moodle intro page, section 5), fill in the three lines below and run the cell. It asks for a GitLab access token (GitLab: user settings → Access tokens, scopes `read_repository` and `write_repository`); the token is kept only for this session and is never committed.

The cell copies the kit, the course code it set up and `results/` into your repository (without the test recordings), commits, pushes and prints your **baseline commit** for the proposal. Then it switches into your repository: from now on you work there and commit and push your changes from there.

**Every later Colab session** (every group member): open this notebook again, run the first cell, then this cell. It clones your repository and switches into it; nothing is copied again.

In [ ]:
GITLAB_URL = ''   # your group's repository, e.g. 'https://gitlab.tu-ilmenau.de/<user>/ac-ws2627-p05b'
GIT_NAME = ''     # your name, for your commits
GIT_EMAIL = ''    # your university e-mail address
if GITLAB_URL:
    import getpass, gitlab_repo
    k = gitlab_repo.use_repository(GITLAB_URL, getpass.getpass('GitLab access token: '), GIT_NAME, GIT_EMAIL)
else:
    print('Fill in GITLAB_URL, GIT_NAME and GIT_EMAIL, then run this cell (needed before the proposal deadline).')